In [1]:
from sensor_core import SensorManager
import numpy as np
import os, time

_real_remove = os.remove
_real_replace = os.replace

def safe_remove(path, retries=50, delay=0.01):
    for _ in range(retries):
        try:
            _real_remove(path)
            return
        except PermissionError:
            time.sleep(delay)

def safe_replace(src, dst, retries=50, delay=0.01):
    for _ in range(retries):
        try:
            _real_replace(src, dst)
            return
        except PermissionError:
            time.sleep(delay)

os.remove = safe_remove
os.replace = safe_replace

print("✅ Windows-safe seal handling enabled")

for f in ["serial_stream_a.bin.seal", "serial_stream_b.bin.seal"]:
    try:
        os.remove(f)
    except FileNotFoundError:
        pass
# USER DEFINED PARAMS
ser_channel_key = ["red", "infrared", "violet"]                    # Names of serial channels
plot_channel_key = [["red", "infrared", "violet"]]                 # Names and gridplot arrangement of plot channels
commport = None                                                    # Name of comm port
baudrate = None   

for f in ["serial_stream_a.bin.seal", "serial_stream_b.bin.seal"]:
    try:
        os.remove(f)
    except FileNotFoundError:
        pass
# Custom Serial Acquisition Function
def acquire_data(*, ser=None, frame_shape=None, **kwargs):
    """ serial acquisition function handler. All parameters are internally generated
    Must include the following parameters:
    :param ser: serial port
    :param window_size: number of frames to capture before passing to shared memory object. Default 1
    :param num_channel: number of channels to capture. 
    """
    channel_data = np.random.rand(frame_shape[1], frame_shape[2]).astype(np.float32)
    return channel_data
# Instantiate Sensor Manager class
sm = SensorManager(ser_channel_key=ser_channel_key,
                   plot_channel_key=plot_channel_key,
                   commport=commport, 
                   baudrate=baudrate, 
                   dtype=np.float32,
                   data_mode='line',
                   frame_shape = (100, 10, 3), # num_points, window_size, num_channels
                   rotate_seconds=5.0,
                   start_stream_ingest=True)# Create 'update process' to handle acquired data
update_process = sm.update_data_process(save_data=False, 
                            filepath='./serial_db.sqlite3',
                            virtual_ser_port = True,
                            func=acquire_data)
# Create 'update process' to handle acquired data
update_process = sm.update_data_process(save_data=False, 
                            filepath='./serial_db.sqlite3',
                            virtual_ser_port = True,
                            func=acquire_data)
# Create 'update process' to handle acquired data
update_process = sm.update_data_process(save_data=False, 
                            filepath='./serial_db.sqlite3',
                            virtual_ser_port = True,
                            func=acquire_data)

✅ Windows-safe seal handling enabled


In [2]:
class BandwidthMeter:
    def __init__(self, sm):
        self.sm = sm
        self.prev_bytes = None
        self.prev_time = None

    def writer_MBps(self):
        m = self.sm.get_metrics()["writer"]
        now = time.time()
        total_bytes = m.get("writer_total_bytes", 0)

        if self.prev_bytes is None:
            self.prev_bytes = total_bytes
            self.prev_time = now
            return 0.0

        bw = (total_bytes - self.prev_bytes) / (now - self.prev_time) / 1e6
        self.prev_bytes = total_bytes
        self.prev_time = now
        return bw

In [13]:
results = []
variables = []
bw = BandwidthMeter(sm)
for firstVar in [50,100,150,200]:
    for secondVar in [5,10,20,40]:
        for thirdVar in [1,2,3,4]:
            for rotateFrame in [2**10,2**11,2**12,2**13,2**14,2**15]:
                for rotateSeconds in [3,4,5,6,7]:
                    time.sleep(0.1)
                    sm.frame_shape = (firstVar, secondVar, thirdVar)
                    sm.rotate_frames = rotateFrame
                    sm.rotate_seconds = rotateSeconds
                    results.append(bw.writer_MBps())
                    variables.append([[firstVar,secondVar,thirdVar],[rotateFrame],[rotateSeconds]])
print(max(results))
print(variables[results.index(max(results))])

27.463857903940852
[[100, 40, 3], [2048], [7]]


In [4]:
sm.get_metrics()

{'writer': {'init': True,
  'writer_alive': True,
  'writer_start_unix': 1770394022.661208,
  'writer_watch_bins': ['C:\\Users\\lukep\\Downloads\\sensor_core2\\sensor_core\\examples\\serial_stream_a.bin',
   'C:\\Users\\lukep\\Downloads\\sensor_core2\\sensor_core\\examples\\serial_stream_b.bin'],
  'writer_watch_seals': ['C:\\Users\\lukep\\Downloads\\sensor_core2\\sensor_core\\examples\\serial_stream_a.bin.seal',
   'C:\\Users\\lukep\\Downloads\\sensor_core2\\sensor_core\\examples\\serial_stream_b.bin.seal'],
  'writer_ring': {'name': '/sensor_ring',
   'capacity': 4096,
   'shape': (100, 10, 3),
   'dtype': 'float32'},
  'writer_active_bin': 'C:\\Users\\lukep\\Downloads\\sensor_core2\\sensor_core\\examples\\serial_stream_a.bin',
  'writer_total_frames': 575902723,
  'writer_total_bytes': 32250552488,
  'writer_rotations': 70302,
  'writer_fps_estimate': 264258.0646173515,
  'writer_last_rotation_unix': 1770398700.2358441,
  'writer_updated_unix': 1770398700.2358441,
  'writer_seal_exi